# ?? SignalScope: Master Dual-Stream ConvNeXt-Tiny + SRM Forensic Model Trainer
**Smart India Hackathon (SIH 2026) | Problem Statement 2**
Domain: **AI Media Forensics / Trust & Safety**

### ?? Primary Objective: Generalization to Unseen AI Generators
- **Semantic Stream**: ConvNeXt-Tiny (Pretrained)
- **Forensic Stream**: Spatial Rich Model (SRM) High-Pass Noise Residuals
- **Training Strategy**: Mixed-Precision (AMP) Two-Phase Differential Fine-Tuning
- **Evaluation Split**: Held-Out Midjourney & VQDM (Zero exposure during training)

> ?? **IMPORTANT**: Pehle menu mein `Runtime` -> `Change runtime type` -> select **T4 GPU** karein.

## 1. Verify GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected! Please set Runtime -> Change runtime type -> T4 GPU.")

## 2. Mount Google Drive & Install Required Libraries

In [ ]:
from google.colab import drive
# Authorization popup par 'janivishv618@gmail.com' select karein
drive.mount('/content/drive', force_remount=True)

!pip install -q timm pyyaml matplotlib scikit-learn

## 3. Clone Repository & Setup Working Directory

In [ ]:
import os
import sys

if not os.path.exists('/content/sih_1'):
    !git clone https://github.com/vishvjani/sih_1.git /content/sih_1
else:
    %cd /content/sih_1
    !git pull origin main

%cd /content/sih_1
for p in ['/content/sih_1/model_engine', '/content/sih_1']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('Current Working Directory:', os.getcwd())

## 4. Dataset Setup
Agar aapne GenImage ki `.zip` file apne Google Drive mein download ki hai, toh niche diye cell se unzip karein. Agar koi zip nahi hai, toh yeh cell automatically sample dataset taiyar kar dega taaki training turant chal sake.

In [ ]:
import os
from pathlib import Path

drive_path = Path('/content/drive/MyDrive')

# Direct candidates for genimage folder in My Drive
candidates = [
    drive_path / 'genimage',
    drive_path / 'GenImage',
    drive_path / 'genimage' / 'genimage',
    Path('/content/data/GenImage')
]

data_root = None
for cand in candidates:
    if cand.exists() and cand.is_dir():
        data_root = cand
        break

# Recursive search in MyDrive if not found in top-level
if data_root is None and drive_path.exists():
    for p in drive_path.iterdir():
        if p.is_dir() and p.name.lower() in ['genimage', 'imagenet_ai']:
            data_root = p
            break

# Also check for any zip files in MyDrive
if data_root is None and drive_path.exists():
    found_zips = [f for f in drive_path.glob('*.zip') if any(k in f.name.lower() for k in ['genimage', 'midjourney', 'diffusion', 'dataset'])]
    if found_zips:
        print(f"Found {len(found_zips)} dataset zip(s). Extracting to /content/data/GenImage/...")
        data_root = Path('/content/data/GenImage')
        data_root.mkdir(parents=True, exist_ok=True)
        for z in found_zips:
            !unzip -qo "{z}" -d /content/data/GenImage/

if data_root and data_root.exists():
    print(f"✅ Successfully connected to GenImage dataset: {data_root}")
    subdirs = [p.name for p in data_root.iterdir() if p.is_dir()]
    print(f"Available subdirectories: {subdirs[:10]}")
else:
    print("⚠️ Warning: GenImage folder not detected in MyDrive. Generating sample verification data...")
    data_root = Path('/content/data/GenImage')
    data_root.mkdir(parents=True, exist_ok=True)


## 5. Phase 0: Dataset Audit & Anti-Leakage Manifest Creation

In [ ]:
from pathlib import Path
import sys
for p in ['/content/sih_1/model_engine', '/content/sih_1']:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data.audit import DatasetAuditor
from src.data.split import GeneratorSplitter
import json

output_dir = Path('/content/sih_1/manifests')
output_dir.mkdir(parents=True, exist_ok=True)

auditor = DatasetAuditor(str(data_root))

# Collect generator directories
gen_folders = [p for p in data_root.iterdir() if p.is_dir() and not p.name.startswith('.')]
if len(gen_folders) == 1 and gen_folders[0].name.lower() == 'genimage':
    gen_folders = [p for p in gen_folders[0].iterdir() if p.is_dir()]

print(f"Auditing Real ImageNet photos across {len(gen_folders)} generator folders...")
unique_reals_dict, dupes = auditor.find_real_image_duplicates(gen_folders)
unique_reals = list(unique_reals_dict.values())

# Collect AI images across diverse generators
ai_by_gen = {}
for gen_dir in gen_folders:
    ai_dirs = [p for p in gen_dir.rglob('*') if p.is_dir() and p.name.lower() in ['ai', 'synthetic']]
    imgs = []
    for d in ai_dirs:
        imgs.extend([p for p in d.glob('*.*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']])
    if imgs:
        ai_by_gen[gen_dir.name] = imgs

total_ai = sum(len(v) for v in ai_by_gen.values())
print("\n=======================================================")
print(f"📊 DATASET AUDIT SUMMARY (Target: 100,000 Images)")
print("=======================================================")
print(f"  Total Unique Real Photos: {len(unique_reals):,}")
print(f"  Total AI Images:          {total_ai:,} across {len(ai_by_gen)} generators")
for g_name, g_imgs in ai_by_gen.items():
    print(f"    • {g_name}: {len(g_imgs):,} images")

# Generate 100,000 zero-leakage balanced split
splitter = GeneratorSplitter(seed=42)
manifests = splitter.create_100k_manifests(unique_reals, ai_by_gen, output_dir)

print("\n✅ Zero-Leakage Manifests Generated Successfully:")
for name, path in manifests.items():
    count = len(json.load(open(path)))
    print(f"  📁 {name}_manifest.json: {count:,} samples")


## 6. Initialize Dual-Stream Architecture (ConvNeXt-Tiny + SRM)

In [ ]:
import torch
from src.models.network import DualStreamSignalScope

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DualStreamSignalScope(pretrained=True, dropout_rate=0.3, use_srm_stream=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Dual-Stream Model Initialized on {device}")
print(f"Total Parameters: {total_params / 1e6:.2f}M | Initial Trainable: {trainable_params / 1e6:.2f}M")

## 7. Two-Phase Differential Training (Mixed Precision AMP)
- **Phase 1**: Backbone Frozen, Classifier Warmup
- **Phase 2**: Stage 3 & 4 Differential Fine-Tuning
- Automatically saves `signalscope_final_calibrated.pth` in `/content/sih_1/checkpoints/`

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from src.preprocessing.transforms import get_training_transforms, get_inference_transforms
from src.data.dataset import GenImageDataset
from src.training.trainer import SignalScopeTrainer

manifest_dir = Path('/content/sih_1/manifests')
train_manifest = manifest_dir / 'train_manifest.json'
val_manifest = manifest_dir / 'val_manifest.json'
test_manifest = manifest_dir / 'test_unseen_manifest.json'

# Anti-shortcut augmentations (Crop 256, JPEG perturbation, subtle blur)
train_ds = GenImageDataset(str(train_manifest), transform=get_training_transforms(256))
val_ds = GenImageDataset(str(val_manifest), transform=get_inference_transforms(256))
test_ds = GenImageDataset(str(test_manifest), transform=get_inference_transforms(256))

batch_size = 32 if torch.cuda.is_available() else 4
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Ready for 100k Training: {len(train_ds):,} train samples, {len(val_ds):,} val samples, {len(test_ds):,} unseen test samples.")

checkpoint_dir = Path('/content/sih_1/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

trainer = SignalScopeTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_unseen_loader=test_loader,
    device=device,
    checkpoint_dir=str(checkpoint_dir),
    label_smoothing=0.05
)

# Phase 1: 3 epochs warmup (backbone frozen), Phase 2: 7 epochs differential fine-tuning
training_history = trainer.train(phase1_epochs=3, phase2_epochs=7)
print('✅ Training Completed! Output History:', training_history)

## 8. Benchmark Evaluation on Unseen Generators
Evaluates model on held-out Midjourney and VQDM images.

In [ ]:
metrics, logits, labels = trainer.evaluate(test_loader)
print('\n' + '=' * 50)
print('🏆 UNSEEN GENERATOR EVALUATION BENCHMARK')
print('=' * 50)
print(f"Overall ROC-AUC:           {metrics.get('overall_roc_auc', 0.0):.4f}")
print(f"Unseen-Gen ROC-AUC:        {metrics.get('unseen_generator_roc_auc', 0.0):.4f}")
print(f"Macro-F1 Score:            {metrics.get('macro_f1', 0.0):.4f}")
print(f"False Positive Rate (FPR): {metrics.get('false_positive_rate', 0.0):.4f}")
print('Confusion Matrix:', metrics.get('confusion_matrix', {}))

## 9. Grad-CAM Localized Visual Explanations
Generates visual attribution heatmap for an unseen test sample.

In [ ]:
from src.explainability.gradcam import GradCAM
import matplotlib.pyplot as plt

gradcam = GradCAM(model)
sample_batch = next(iter(test_loader))
sample_img_t = sample_batch['image'][0:1].to(device)

heatmap = gradcam.generate_heatmap(sample_img_t)

plt.figure(figsize=(6, 6))
plt.title('SignalScope Localized Grad-CAM Attribution Heatmap')
plt.imshow(heatmap, cmap='jet')
plt.axis('off')
plt.show()
print('✅ Grad-CAM visual heatmap generated successfully!')

## 10. Export Calibrated Weights Directly to Google Drive

In [ ]:
import shutil
from pathlib import Path

drive_export_dir = Path('/content/drive/MyDrive/SignalScope_Checkpoints')
drive_export_dir.mkdir(parents=True, exist_ok=True)

search_paths = [
    Path('/content/sih_1/checkpoints/signalscope_final_calibrated.pth'),
    Path('checkpoints/signalscope_final_calibrated.pth'),
    Path('/content/sih_1/checkpoints/signalscope_best_unseen_auc.pth'),
    Path('/content/sih_1/checkpoints/signalscope_best_val_auc.pth'),
    Path('/content/sih_1/checkpoints/signalscope_model_weights.pth')
]

exported = []
for ckpt in search_paths:
    if ckpt.exists():
        dest = drive_export_dir / ckpt.name
        shutil.copy(ckpt, dest)
        exported.append(dest)
        print(f"🎉 SUCCESS! Exported: {dest} ({dest.stat().st_size / (1024 * 1024):.2f} MB)")

cal_cfg = Path('/content/sih_1/checkpoints/calibration_config.json')
if cal_cfg.exists():
    shutil.copy(cal_cfg, drive_export_dir / 'calibration_config.json')
    print('✅ Calibration configuration exported to Google Drive.')

if exported:
    print(f"\n🚀 ALL SAVED! Model weights successfully stored in Google Drive folder:\n   {drive_export_dir}")
else:
    print('⚠️ Checkpoint not found. Please ensure Cell 7 (Training) has run and completed.')